## Standard Errors for κ (Kappa)

The standard errors for κ reported in stage 2 are currently not reliable.  
The reason is that stage 2 treats the predictions generated in stage 1 as fixed regressors, even though these predictions are themselves estimated quantities.

This creates a **generated-regressor problem**: the uncertainty from the stage 1 estimation is ignored in stage 2. As a result, the reported standard errors for κ are downward biased and do not correctly reflect the true sampling variability.

To address this issue, we bootstrap the **entire two-stage estimation procedure** rather than treating stage 2 in isolation. In each bootstrap replication, the full workflow is re-estimated: the stage 1 model is refit, new predictions are generated, and stage 2 is re-estimated using these predictions.

This approach propagates the estimation uncertainty from stage 1 into stage 2, thereby accounting for the generated-regressor problem. The resulting bootstrap distribution of κ allows us to construct standard errors and confidence intervals that correctly reflect uncertainty from both stages of the estimation process.


### Moving Block Bootstrap

Classic bootstrapping means:

1. Sample a new dataset by drawing from the training data with replacement until it has the same size as the original dataset.
2. Repeat this many times until you have enough synthetic datasets.
3. Estimate the model on all synthetic datasets.
4. Approximate the distribution of your point estimate by using the variation in point estimates generated on the synthetic data.

Basically, we change the underlying data a bit to see how our estimate would change.
The problem with timeseries data is, that we cannot simply draw observations with replacement becasue adjacent observations/time points are autocorrelated.

In Moving Block Bootstrap (MBB) draws are done over the ‘blocked sets’ of consecutive data, instead of over individual data rows.

In [6]:
import pandas as pd
import numpy as np
import random
from bootstrap import bootstrap_two_stage_block

In [4]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [7]:
# set seed
random.seed(42)

# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx

# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each 
num_topics = len(topic_cols)  
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [8]:
result = bootstrap_two_stage_block(
    X=X, y=y,
    window_size=45, n_lags=4, lambda_val=0.009844,
    B=100,
    block_len=22,          # e.g., ~ 3 months if weekly data
    standardize=True,
    random_state=123
)

Block bootstrap (2-stage):   0%|          | 0/100 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [12]:
print(result["bootstrap_summary"])
print(result["draws"][["kappa", "intercept", "r2_insample_stage2", "r2_oos_stage2"]].describe())

{'B_requested': 100, 'B_success': 100, 'failure_rate': 0.0, 'block_len': 22, 'kappa_point': np.float64(0.8711836382476524), 'kappa_boot_se': 0.2918895730013679, 'kappa_ci_2p5_97p5': (np.float64(7.084629875494349e-07), np.float64(0.9687989755584336)), 'intercept_point': np.float64(0.00039776166984947054), 'intercept_boot_se': 0.00016357864387031397, 'intercept_ci_2p5_97p5': (np.float64(0.00014673882547491576), np.float64(0.0006976455083097795))}
              kappa   intercept  r2_insample_stage2  r2_oos_stage2
count  1.000000e+02  100.000000        1.000000e+02     100.000000
mean   4.493101e-01    0.000415        4.954573e-04      -0.003488
std    2.918896e-01    0.000164        2.711202e-03       0.009661
min    4.347152e-09    0.000118       -1.209670e-07      -0.091323
25%    4.011971e-01    0.000290       -1.244531e-09      -0.003325
50%    5.000000e-01    0.000418       -2.229239e-11      -0.001367
75%    5.115619e-01    0.000523        1.280831e-04      -0.000253
max    9.866207